In [1]:
import sys
import os

import numpy as np
import pandas as pd

from tqdm.notebook import tqdm
from sklearn.mixture import GaussianMixture
from sklearn.cluster import DBSCAN, AgglomerativeClustering, KMeans
from kmodes.kprototypes import KPrototypes
from sklearn.metrics import silhouette_score
from imblearn.over_sampling import SMOTE


src_path = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "preliminar_phase_submision"))
if src_path not in sys.path:
    sys.path.append(src_path)

from team_code import data_cleaning_main
from helper_functions.feature_transformer import *


input_file = "/Users/uche/Sepsis_Hackathon/The-2024-Pediatric-Sepsis-Challenge/preliminar_phase_submision/training_data/SyntheticData_Training.csv"
_ = data_cleaning_main(input_file)
data = pd.read_csv("cleaned_data.csv")
target = "inhospital_mortality"

transformer = FeatureTransformer()
transformer.fit(data)
transformed_df = transformer.transform(data)

In [2]:
feature_types = create_feature_types(
    numerical_features = data.select_dtypes(include=['float']).columns.tolist(),
    binary_category = BINARY_FEATURES,
    nominal_category = NOMINAL_FEATURES,
    ordinal_category = list(ORDINAL_MAPPING.keys()) + list(THREE_VALUE_ORDINAL_MAPPING.keys())
    )

data_without_nominal = transformed_df[[col for col in data.columns if col not in NOMINAL_FEATURES]]
df = pd.concat([data_without_nominal, data[NOMINAL_FEATURES]], axis=1)

In [18]:
def compute_feature_correlations(df, target_col, feature_types, p_value_threshold=0.05):
    df = df.copy()
    results = []
    target = df[target_col]

    for feature, ftype in feature_types.items():
        if feature == target_col:
            continue

        if ftype == 'numerical':
            corr, p_value = pointbiserialr(df[feature], target)

        elif ftype == 'ordinal':
            corr, p_value = spearmanr(df[feature], target)

        elif ftype == 'nominal':
            # Chi-Square Test
            contingency_table = pd.crosstab(df[feature], target)
            chi2, p_value, _, _ = chi2_contingency(contingency_table)
            corr = chi2  # Using chi-square statistic as a measure

        elif ftype == 'binary':
            corr = np.corrcoef(df[feature], target)[0, 1]  # Phi coefficient
            p_value = np.nan  # Not applicable in this case

        else:
            continue

        # Apply significance threshold
        significance = p_value < p_value_threshold if not np.isnan(
            p_value) else True

        results.append({'Feature': feature, 'Type': ftype, 'Correlation': corr if significance else False,
                       'p-value': p_value, 'Significant': significance})

    # Compute Mutual Information for all features
    discrete_features = []
    df.drop(columns=NOMINAL_FEATURES, inplace=True)

    # Iterate over the feature_types dictionary
    for feature, ftype in feature_types.items():
        if feature != target_col:  # Skip the target column
            # Check if the feature type is in ['ordinal', 'binary']
            if ftype in ['ordinal', 'binary', 'nominal']:
                discrete_features.append(True)
            else:
                discrete_features.append(False)
    

    # Now, use the discrete_features list in mutual_info_classif
    mi_scores = mutual_info_classif(
        df.drop(columns=[target_col]), target, discrete_features=discrete_features)
    for i, feature in enumerate(feature_types.keys()):
        if feature == target_col:
            continue
        for res in results:
            if res['Feature'] == feature:
                res['Mutual Information'] = mi_scores[i]
                break

    return pd.DataFrame(results)

In [19]:
compute_feature_correlations(
    df=df,
    target_col=target,
    feature_types=feature_types
)

/Users/uche/Sepsis_Hackathon/venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:2999: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/uche/Sepsis_Hackathon/venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3000: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


IndexError: boolean index did not match indexed array along axis 1; size of axis is 128 but size of corresponding boolean axis is 132

In [31]:
'deliveryloc_adm_Health centre' in BINARY_FEATURES

True

In [20]:
from math import ceil
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import pointbiserialr, spearmanr, chi2_contingency
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

Help on function mutual_info_classif in module sklearn.feature_selection._mutual_info:

mutual_info_classif(X, y, *, discrete_features='auto', n_neighbors=3, copy=True, random_state=None, n_jobs=None)
    Estimate mutual information for a discrete target variable.

    Mutual information (MI) [1]_ between two random variables is a non-negative
    value, which measures the dependency between the variables. It is equal
    to zero if and only if two random variables are independent, and higher
    values mean higher dependency.

    The function relies on nonparametric methods based on entropy estimation
    from k-nearest neighbors distances as described in [2]_ and [3]_. Both
    methods are based on the idea originally proposed in [4]_.

    It can be used for univariate features selection, read more in the
    :ref:`User Guide <univariate_feature_selection>`.

    Parameters
    ----------
    X : {array-like, sparse matrix} of shape (n_samples, n_features)
        Feature matrix.



In [48]:
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif, mutual_info_classif
X = df[list(ORDINAL_MAPPING.keys()) + list(THREE_VALUE_ORDINAL_MAPPING.keys())]
y = df[target]
S = SelectKBest(mutual_info_classif, k=10)
X_new = S.fit_transform(X, y)
S.get_feature_names_out()

array(['oxygenavail_adm', 'bcsmotor_adm', 'bcsverbal_adm',
       'totalbreastfed_adm', 'traveldist_adm', 'feedingstatus_adm',
       'priorweekantimal_adm', 'diarrheaoften_adm', 'tbcontact_adm',
       'momhiv_adm'], dtype=object)

In [44]:
S.get_feature_names_out()

array(['oxygenavail_adm', 'bcsmotor_adm', 'bcsverbal_adm', 'vaccdpt_adm',
       'prioryearcough_adm', 'exclbreastfed_adm', 'bednet_adm',
       'feedingstatus_adm', 'watersource_adm', 'momhiv_adm'], dtype=object)

In [50]:
X = transformed_df[data.select_dtypes(include=['float']).columns.tolist()]
y = transformed_df[target]
S = SelectKBest(mutual_info_classif, k=10)
X_new = S.fit_transform(X, y)
S.get_feature_names_out()

array(['agecalc_adm', 'height_cm_adm', 'weight_kg_adm', 'muac_mm_adm',
       'sysbp_mmhg_adm', 'spo2site1_pc_oxi_adm', 'momage_adm',
       'momagefirstpreg_adm', 'lactate_mmolpl_adm', 'glucose_mmolpl_adm'],
      dtype=object)

In [51]:
X = df[data.select_dtypes(include=['float']).columns.tolist()]
y = df[target]
S = SelectKBest(mutual_info_classif, k=10)
X_new = S.fit_transform(X, y)
S.get_feature_names_out()

array(['agecalc_adm', 'height_cm_adm', 'weight_kg_adm', 'muac_mm_adm',
       'spo2site2_pc_oxi_adm', 'momagefirstpreg_adm', 'alivechildren_adm',
       'deadchildren_adm', 'lactate_mmolpl_adm', 'glucose_mmolpl_adm'],
      dtype=object)

In [63]:
with pd.option_context('display.max_rows', None,):
    display(data[BINARY_FEATURES].T)

,0,1,2,3,4,5,6,7,8,9,...,2138,2139,2140,2141,2142,2143,2144,2145,2146,2147
sex_adm,Female,Male,Female,Female,Female,Male,Male,Female,Female,Male,...,Male,Female,Female,Male,Female,Female,Female,Female,Male,Female
spo2onoxy_adm,No,No,No,No,No,Yes,No,No,No,No,...,Yes,No,No,No,No,No,No,No,No,No
respdistress_adm,No,No,No,No,No,No,No,No,No,No,...,No,No,No,No,Yes,Yes,No,No,No,No
caprefill_adm,No,No,No,No,No,No,No,No,No,No,...,Yes,No,No,No,No,No,No,No,Yes,No
bcseye_adm,Watches or follows,Watches or follows,Watches or follows,Watches or follows,Watches or follows,Watches or follows,Watches or follows,Watches or follows,Watches or follows,Watches or follows,...,Watches or follows,Watches or follows,Fails to watch or follow,Watches or follows,Watches or follows,Watches or follows,Watches or follows,Watches or follows,Watches or follows,Watches or follows
admitabx_adm___1,Checked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Checked,Checked,Unchecked,Checked,...,Unchecked,Unchecked,Unchecked,Checked,Checked,Unchecked,Checked,Checked,Unchecked,Unchecked
admitabx_adm___2,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,...,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked
admitabx_adm___3,Unchecked,Checked,Checked,Unchecked,Unchecked,Checked,Unchecked,Unchecked,Unchecked,Unchecked,...,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Checked
admitabx_adm___4,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,...,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked
admitabx_adm___5,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,...,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked,Unchecked
